In [1]:
!pip install transformers datasets pandas scikit-learn sentencepiece accelerate torch evaluate rouge_score seqeval pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 93.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 100.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import torch
import gc
import os
import json
import re
import ast
from collections import defaultdict
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from tqdm import tqdm
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    DataCollatorWithPadding,
    DataCollatorForTokenClassification,
    pipeline
)

from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import precision_recall_fscore_support, classification_report

In [3]:
from huggingface_hub import login
# login("YOUR_TOKEN")

In [4]:
# ==========================================
# KONFIGURASI
# ==========================================
CSV_FILE_PATH = "final_processed_dataset (1).csv"
BASE_OUTPUT_DIR = "./separated_models"
MARKET_CATEGORIES = ['market', 'stock', 'finance', 'corporate-action', 'business', 'macroeconomy']

In [5]:
def cleanup_memory():
    """Membersihkan VRAM GPU setelah training satu model selesai"""
    torch.cuda.empty_cache()
    gc.collect()

cleanup_memory()

In [6]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import DatasetDict, Dataset

def clean_and_normalize_text(text):
    """
    Membersihkan dan menormalisasi text dengan menghilangkan tag, prefix, dan normalisasi lainnya
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Hilangkan prefix yang umum
    prefixes_to_remove = [
        r'^KOMPAS\.com\s*[-–—]\s*',  # KOMPAS.com -, KOMPAS.com –, dll
        r'^JAKARTA,\s*KOMPAS\.com\s*[-–—]\s*',  # JAKARTA, KOMPAS.com -
        r'^DENPASAR,\s*KOMPAS\.com\s*[-–—]\s*',  # DENPASAR, KOMPAS.com -
        r'^BANDUNG,\s*KOMPAS\.com\s*[-–—]\s*',   # BANDUNG, KOMPAS.com -
        r'^SURABAYA,\s*KOMPAS\.com\s*[-–—]\s*',  # SURABAYA, KOMPAS.com -
        r'^YOGYAKARTA,\s*KOMPAS\.com\s*[-–—]\s*', # YOGYAKARTA, KOMPAS.com -
        r'^MEDAN,\s*KOMPAS\.com\s*[-–—]\s*',     # MEDAN, KOMPAS.com -
        r'^MAKASSAR,\s*KOMPAS\.com\s*[-–—]\s*',  # MAKASSAR, KOMPAS.com -
        r'^SEMARANG,\s*KOMPAS\.com\s*[-–—]\s*',  # SEMARANG, KOMPAS.com -
        r'^PALEMBANG,\s*KOMPAS\.com\s*[-–—]\s*', # PALEMBANG, KOMPAS.com -
        r'^BALIKPAPAN,\s*KOMPAS\.com\s*[-–—]\s*', # BALIKPAPAN, KOMPAS.com -
        r'^BANJARMASIN,\s*KOMPAS\.com\s*[-–—]\s*', # BANJARMASIN, KOMPAS.com -
        r'^MANADO,\s*KOMPAS\.com\s*[-–—]\s*',    # MANADO, KOMPAS.com -
        r'^JAYAPURA,\s*KOMPAS\.com\s*[-–—]\s*',  # JAYAPURA, KOMPAS.com -
        r'^MATARAM,\s*KOMPAS\.com\s*[-–—]\s*',   # MATARAM, KOMPAS.com -
        r'^KUPANG,\s*KOMPAS\.com\s*[-–—]\s*',    # KUPANG, KOMPAS.com -
        r'^PADANG,\s*KOMPAS\.com\s*[-–—]\s*',    # PADANG, KOMPAS.com -
        r'^PONTIANAK,\s*KOMPAS\.com\s*[-–—]\s*', # PONTIANAK, KOMPAS.com -
        r'^BANDA ACEH,\s*KOMPAS\.com\s*[-–—]\s*', # BANDA ACEH, KOMPAS.com -
        r'^PALU,\s*KOMPAS\.com\s*[-–—]\s*',      # PALU, KOMPAS.com -
        r'^AMBON,\s*KOMPAS\.com\s*[-–—]\s*',     # AMBON, KOMPAS.com -
        r'^TERNATE,\s*KOMPAS\.com\s*[-–—]\s*',   # TERNATE, KOMPAS.com -
        r'^SORONG,\s*KOMPAS\.com\s*[-–—]\s*',    # SORONG, KOMPAS.com -
        r'^KENDARI,\s*KOMPAS\.com\s*[-–—]\s*',   # KENDARI, KOMPAS.com -
        r'^GORONTALO,\s*KOMPAS\.com\s*[-–—]\s*', # GORONTALO, KOMPAS.com -
        r'^MAMUJU,\s*KOMPAS\.com\s*[-–—]\s*',    # MAMUJU, KOMPAS.com -
        r'^TANJUNGSELOR,\s*KOMPAS\.com\s*[-–—]\s*', # TANJUNGSELOR, KOMPAS.com -
        r'^MERAUKE,\s*KOMPAS\.com\s*[-–—]\s*',   # MERAUKE, KOMPAS.com -
    ]
    
    cleaned_text = text
    for pattern in prefixes_to_remove:
        cleaned_text = re.sub(pattern, '', cleaned_text, flags=re.IGNORECASE)
    
    # 2. Hilangkan tag HTML sederhana
    cleaned_text = re.sub(r'<[^>]+>', '', cleaned_text)
    
    # 3. Normalisasi whitespace
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)  # Ganti multiple spaces dengan single space
    cleaned_text = cleaned_text.strip()
    
    # 4. Normalisasi dash dan hyphen
    cleaned_text = re.sub(r'[–—]', '-', cleaned_text)
    
    # 5. Hilangkan karakter khusus yang tidak perlu di awal text
    cleaned_text = re.sub(r'^[^\w\s]+', '', cleaned_text)
    
    # 6. Normalisasi quotes
    cleaned_text = re.sub(r'[`´]', "'", cleaned_text)
    cleaned_text = re.sub(r'[«»""]', '"', cleaned_text)
    
    # 7. Hilangkan bullet points dan numbering di awal
    cleaned_text = re.sub(r'^[\s•·▪➢➤›-]*\d+[\.\)]\s*', '', cleaned_text)
    cleaned_text = re.sub(r'^[\s•·▪➢➤›-]+\s*', '', cleaned_text)
    
    # 8. Normalisasi ellipsis
    cleaned_text = re.sub(r'\.{3,}', '...', cleaned_text)
    
    # 9. Hilangkan extra spaces setelah normalisasi
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def apply_category_mapping(df):
    """
    Apply category mapping ke dataframe
    """
    df_mapped = df.copy()
    df_mapped["category_mapped"] = df_mapped["category"].apply(map_category)
    return df_mapped

def preprocess_dataframe(df, apply_text_cleaning=True, apply_category_mapping_flag=True):
    """
    Preprocess lengkap untuk dataframe
    """
    print("Memulai preprocessing data...")
    
    # 1. Hilangkan rows dengan data kosong
    initial_count = len(df)
    df = df.dropna(subset=['body_text', 'category', 'ner', 'summary'])
    after_dropna_count = len(df)
    print(f"  - Drop NA: {initial_count} → {after_dropna_count} rows")
    
    # 2. Filter text yang terlalu pendek
    df = df[df['body_text'].str.len() > 20]
    after_length_filter = len(df)
    print(f"  - Filter length >20: {after_dropna_count} → {after_length_filter} rows")
    
    # 3. Cleaning text
    if apply_text_cleaning:
        print("  - Membersihkan dan menormalisasi text...")
        df_cleaned = df.copy()
        df_cleaned['body_text_original'] = df_cleaned['body_text']  # Simpan original
        df_cleaned['body_text'] = df_cleaned['body_text'].apply(clean_and_normalize_text)
        
        # Hitung perubahan
        changed_count = (df_cleaned['body_text_original'] != df_cleaned['body_text']).sum()
        print(f"    {changed_count}/{len(df_cleaned)} texts berubah setelah cleaning")
        
        # Filter lagi yang menjadi terlalu pendek setelah cleaning
        before_clean_filter = len(df_cleaned)
        df_cleaned = df_cleaned[df_cleaned['body_text'].str.len() > 10]
        after_clean_filter = len(df_cleaned)
        print(f"    Filter setelah cleaning: {before_clean_filter} → {after_clean_filter} rows")
        
        df = df_cleaned
    
    # 4. Apply category mapping
    if apply_category_mapping_flag:
        print("  - Menerapkan category mapping...")
        df = apply_category_mapping(df)
        
        # Tampilkan distribusi kategori
        category_dist = df['category_mapped'].value_counts()
        print(f"    Distribusi kategori setelah mapping:")
        for cat, count in category_dist.head(10).items():  # Tampilkan 10 teratas
            print(f"      {cat}: {count} samples")
        if len(category_dist) > 10:
            print(f"      ... dan {len(category_dist) - 10} kategori lainnya")
    
    print(f"✅ Preprocessing selesai. Total data: {len(df)} rows")
    return df

# Load Dataset dengan preprocessing
print("Loading Dataset dengan preprocessing...")
df_raw = pd.read_csv(CSV_FILE_PATH)

# Preprocess data
df_processed = preprocess_dataframe(
    df_raw, 
    apply_text_cleaning=True, 
    apply_category_mapping_flag=False
)

# Split data
train_val_df, test_df = train_test_split(df_processed, test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=42)

print(f"\nJumlah Data setelah preprocessing:")
print(f"  Train : {len(train_df)} baris")
print(f"  Val   : {len(val_df)} baris") 
print(f"  Test  : {len(test_df)} baris")

# Buat datasets
raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True))
})

# Tampilkan contoh sebelum dan sesudah cleaning
print("\n📋 CONTOH SEBELUM DAN SESUDAH CLEANING:")
print("=" * 80)
sample_idx = 0
if 'body_text_original' in train_df.columns:
    original = train_df.iloc[sample_idx]['body_text_original']
    cleaned = train_df.iloc[sample_idx]['body_text']
    
    print("SEBELUM:")
    print(f"  Text: {original[:200]}...")
    print("\nSESUDAH:")
    print(f"  Text: {cleaned[:200]}...")
    print("=" * 80)

Loading Dataset dengan preprocessing...
Memulai preprocessing data...
  - Drop NA: 3116 → 3116 rows
  - Filter length >20: 3116 → 3116 rows
  - Membersihkan dan menormalisasi text...
    3094/3116 texts berubah setelah cleaning
    Filter setelah cleaning: 3116 → 3116 rows
✅ Preprocessing selesai. Total data: 3116 rows

Jumlah Data setelah preprocessing:
  Train : 2523 baris
  Val   : 281 baris
  Test  : 312 baris

📋 CONTOH SEBELUM DAN SESUDAH CLEANING:
SEBELUM:
  Text: KOMPAS.com -Pelatih tim nasional Italia, Gennaro Gattuso, angkat bicara usai hasil drawing playoff Piala Dunia 2026 Zona Eropa mempertemukan timnya dengan Irlandia Utara.

Drawing playoffPiala Dunia 2...

SESUDAH:
  Text: Pelatih tim nasional Italia, Gennaro Gattuso, angkat bicara usai hasil drawing playoff Piala Dunia 2026 Zona Eropa mempertemukan timnya dengan Irlandia Utara. Drawing playoffPiala Dunia 2026Zona Eropa...


In [7]:
train_df.tail(10)

,author,body_text,category,channel,error,link,published_date,scraped_at,title,ner,summary,body_text_original
1104,Tim Cek Fakta,"Berdasarkan verifikasi Kompas.com sejauh ini, ...",news,cek_fakta,NaN,https://www.kompas.com/cekfakta/read/2025/11/2...,"Kompas.com, 22 November 2025, 10:01 WIB",2025-11-22T16:55:31.011783,[KLARIFIKASI] Video Ini Bukan Pengerahan Kapal...,"[{'entity': 'Kompas.com', 'type': 'ORG'}, {'en...",Video yang beredar mengenai kapal perang China...,"Berdasarkan verifikasi Kompas.com sejauh ini, ..."
1036,NaN,Penjabat (Pj) Gubernur Provinsi Sulawesi Selat...,news,kilas_daerah,NaN,https://kilasdaerah.kompas.com/sulsel/read/202...,"Kompas.com- 26/05/2024, 13:02 WIB",2025-11-22T16:55:28.875523,"Budayakan Hidup Sehat, Pj Gubernur Sulsel Ajak...","[{'entity': 'Zudan Arif Fakrulloh', 'type': 'P...",Pj Gubernur Sulsel Zudan Arif Fakrulloh ajak O...,KOMPAS.com– Penjabat (Pj) Gubernur Provinsi Su...
421,NaN,Pani Gold Project(PGP) atau Proyek Emas Pani a...,news,kilas_korporasi,NaN,https://kilaskorporasi.kompas.com/bakti-merdek...,"Kompas.com- 23/12/2024, 16:03 WIB",2025-11-22T16:54:57.229858,"Pani Gold Project, Proyek Masa Depan MDKA yang...","[{'entity': 'Pani Gold Project', 'type': 'ORG'...","Pani Gold Project di Gorontalo, milik PT Merde...",KOMPAS.com-Pani Gold Project(PGP) atau Proyek ...
220,Sakina Rakhma Diah Setiawan,"PT Bukit Makmur Mandiri Utama (BUMA), anak usa...",finance,energi,NaN,https://money.kompas.com/read/2025/11/19/14561...,"Kompas.com, 19 November 2025, 14:56 WIB",2025-11-22T16:54:45.558127,"Anak Usaha DOID Lunasi Senior Notes 2026 212,2...","[{'entity': 'JAKARTA', 'type': 'GPE'}, {'entit...",BUMA melunasi lebih awal Senior Notes 2026 sen...,"JAKARTA, KOMPAS.com —PT Bukit Makmur Mandiri U..."
2980,Adhi Prasetya -Sport,Menteri Pemuda dan Olahraga Erick Thohir memas...,sport-lain,sport,NaN,https://sport.detik.com/sport-lain/d-8222781/s...,"Jumat, 21 Nov 2025 23:15 WIB",2025-11-22T13:04:37.750168,SEA Games 2025: CdM Optimis Indonesia Penuhi T...,"[{'entity': 'Erick Thohir', 'type': 'PER'}, {'...",Indonesia menargetkan 80 emas di SEA Games 202...,Menteri Pemuda dan Olahraga Erick Thohir memas...
2292,"Aisyah Sekar Ayu Maharani,",Sertifikat tanah yang hilang diimbau untuk seg...,properti,tips_properti,NaN,https://properti.kompas.com/read/2025/11/22/13...,"Kompas.com- 22 November 2025, 13:00 WIB",2025-11-22T16:55:54.917453,Berapa Biaya Urus Sertifikat Tanah yang Hilang?,"[{'entity': 'JAKARTA', 'type': 'GPE'}, {'entit...",Masyarakat yang kehilangan sertifikat tanah di...,"JAKARTA, KOMPAS.com- Sertifikat tanah yang hil..."
1259,"Suhaiela Bahfein,",Bank Indonesia (BI) mengungkap alasan penerima...,properti,properti,NaN,https://www.kompas.com/properti/read/2025/11/2...,"Kompas.com, 20 November 2025, 23:00 WIB",2025-11-22T16:55:39.777970,"Ini Alasan BI Terima REPO SMF, Jadi Sejarah Baru","[{'entity': 'Bank Indonesia', 'type': 'ORG'}, ...",BI menerima corporate bond SMF sebagai underly...,"JAKARTA, KOMPAS.com- Bank Indonesia (BI) mengu..."
2462,Anisa Indraini -detikFinance,Menteri Keuangan Purbaya Yudhi Sadewa buka sua...,finance,berita-ekonomi-bisnis,NaN,https://finance.detik.com/berita-ekonomi-bisni...,"Sabtu, 22 Nov 2025 12:45 WIB",2025-11-22T12:50:42.875305,Purbaya Ungkap Anak Buahnya Dipanggil Kejagung...,"[{'entity': 'Purbaya Yudhi Sadewa', 'type': 'P...",Menkeu Purbaya menyebut pencekalan mantan Dirj...,Menteri Keuangan Purbaya Yudhi Sadewa buka sua...
2806,Andi Hidayat -detikFinance,Kementerian Keuangan (Kemenkeu) menyebut Indon...,finance,bursa-dan-valas,NaN,https://finance.detik.com/bursa-dan-valas/d-82...,"Senin, 17 Nov 2025 18:30 WIB",2025-11-22T12:59:54.882625,"RI Surplus 1 Miliar Ton Karbon Kredit, Kemenke...","[{'entity': 'Kementerian Keuangan', 'type': 'O...",Indonesia dinilai lebih progresif dalam meneka...,Kementerian Keuangan (Kemenkeu) menyebut Indon...
2714,Hana Nushratu -detikFinance,Dompet digital DANA menghadirkan fitur jaminan...,finance,fintech,NaN,https://finance.detik.com/fintech/d-8

In [8]:
# ENTITY TYPE MAPPING CONFIGURATION
ENTITY_TYPE_MAPPING = {
    # Mapping dari entity types di data ke standard types
    "AGE": "QTY",      # Age -> Quantity
    "BEA": "PRD",      # Beast -> Product  
    "COL": "ORG",      # College -> Organization
    "CULT": "REG",     # Culture -> Religion
    "CUR": "MON",      # Currency -> Money
    "DAY": "DAT",      # Day -> Date
    "HOT": "FAC",      # Hotel -> Facility
    "LA": "LAW",       # Law -> Law Entity
    "LAN": "LAN",      # Language -> Language (same)
    "LAW": "LAW",      # Law -> Law Entity (same)
    "LEAGUE": "ORG",   # League -> Organization
    "NOR": "ORG",      # Political Organization -> Organization
    "ORD": "ORD",      # Ordinal -> Ordinal (same)
    "ORF": "ORG",      # Organization Formal -> Organization
    "ORG": "ORG",      # Organization -> Organization (same)
    "PER": "PER",      # Person -> Person (same)
    "PLAN": "EVT",     # Plan -> Event
    "POL": "ORG",      # Political -> Organization
    "POR": "ORG",      # Portfolio -> Organization
    "PRC": "PRC",      # Percent -> Percent (same)
    "PRD": "PRD",      # Product -> Product (same)
    "PRO": "ORG",      # Professional -> Organization
    "PROC": "EVT",     # Process -> Event
    "PROD": "PRD",     # Product -> Product (same)
    "QTY": "QTY",      # Quantity -> Quantity (same)
    "RAT": "PRC",      # Rate -> Percent
    "REG": "REG",      # Region -> Geopolitical Entity
    "REL": "REG",      # Religion -> Religion
    "TECH": "PRD",     # Technology -> Product
    "TIM": "TIM",      # Time -> Time (same)
    "URL": "WEB",      # URL -> Web
    "WAA": "WOA",      # Work of Art -> Work of Art
    "WAO": "WOA",      # Work of Art -> Work of Art
    "WEB": "WEB",      # Web -> Web (same)
    "WGA": "WOA",      # Work of Art -> Work of Art
    "WOA": "WOA",      # Work of Art -> Work of Art (same)
    "WOK": "WOA",      # Work of Art -> Work of Art
    "WOP": "WOA",      # Work of Art -> Work of Art
    "WP": "WOA",       # Work of Art -> Work of Art
    "WPA": "WOA",      # Work of Art -> Work of Art
    "WQA": "WOA",      # Work of Art -> Work of Art
    
    # Standard types dari prompt
    "CRD": "CRD",
    "DAT": "DAT", 
    "EVT": "EVT",
    "FAC": "FAC",
    "GPE": "GPE",
    "LOC": "LOC",
    "MON": "MON",
}

# Target entity types sesuai prompt
TARGET_ENTITY_TYPES = {
    "CRD", "DAT", "EVT", "FAC", "GPE", "LAW", "LOC", "MON", 
    "NOR", "ORD", "ORG", "PER", "PRC", "PRD", "QTY", "REG", 
    "TIM", "WOA", "LAN"
}

def map_entity_type(entity_type):
    """Map entity type ke standard type"""
    mapped = ENTITY_TYPE_MAPPING.get(entity_type, entity_type)
    if mapped not in TARGET_ENTITY_TYPES:
        return "ORG"
    return mapped

def norm_type(t):
    """Normalize type dengan mapping"""
    original_norm = t.upper().replace(" ", "_").replace("/", "_")
    return map_entity_type(original_norm)

# TRAIN NER MODEL
NER_MODEL_CANDIDATES = [
    # "cahya/bert-base-indonesian-522M",
    # "indobenchmark/indobert-base-p1",
    "cahya/NusaBert-ner-v1.3"
]

def parse_ner_field(raw):
    if raw is None:
        return []
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        return [raw]
    if isinstance(raw, str):
        s = raw.strip()
        if not s:
            return []
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list):
                return obj
            if isinstance(obj, dict):
                return [obj]
            return []
        except Exception:
            return []
    return []

def extract_entity_types(example):
    ents = parse_ner_field(example["ner"])
    types = [e.get("type", "") for e in ents if "type" in e]
    return {"ent_types": types}

def train_ner_model(model_name, raw_datasets, base_output_dir, max_length=512, batch_size=16, epochs=5):
    print(f"Training NER model: {model_name}")
    print("=" * 50)
    
    try:
        tmp = raw_datasets.map(extract_entity_types)
        
        all_types = set()
        for split in ["train", "validation"]:
            for ts in tmp[split]["ent_types"]:
                all_types.update(ts)
        
        normalized_types = {norm_type(t) for t in all_types if t}
        
        filtered_types = {t for t in normalized_types if t in TARGET_ENTITY_TYPES}
        
        print(f"Original entity types: {len(all_types)}")
        print(f"After normalization: {len(normalized_types)}")
        print(f"After filtering to target types: {len(filtered_types)}")
        
        final_types = filtered_types if filtered_types else normalized_types
        
        label_list = sorted(
            {f"B-{t}" for t in final_types}
            | {f"I-{t}" for t in final_types}
            | {"O"}
        )
        label_to_id = {l: i for i, l in enumerate(label_list)}
        id_to_label = {i: l for l, i in label_to_id.items()}
        
        print(f"Final entity types: {len(final_types)}")
        print(f"Total labels: {len(label_list)}")
        print(f"Labels: {sorted(final_types)}")
        
        tokenizer_ner = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        
        model_ner = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(label_list),
            id2label=id_to_label,
            label2id=label_to_id,
            ignore_mismatched_sizes=True
        )
        
        def find_entity_spans(text, entities):
            spans = []
            for ent in entities:
                ent_text = (ent.get("entity") or "").strip()
                ent_type_raw = ent.get("type")
                if not ent_text or not ent_type_raw:
                    continue

                # Gunakan mapped entity type
                ent_type = norm_type(ent_type_raw)
                pattern = re.escape(ent_text)
                for m in re.finditer(pattern, text):
                    start, end = m.start(), m.end()
                    spans.append((start, end, ent_type))
            return spans

        def preprocess_ner(examples):
            texts = examples["body_text"]
            ner_raw_list = examples["ner"]

            encodings = tokenizer_ner(
                texts,
                truncation=True,
                padding="max_length",
                max_length=max_length,
                return_offsets_mapping=True,
            )

            all_labels = []

            for i, text in enumerate(texts):
                ents = parse_ner_field(ner_raw_list[i])
                spans = find_entity_spans(text, ents)
                offsets = encodings["offset_mapping"][i]
                labels = ["O"] * len(offsets)

                for (start_char, end_char, ent_type) in spans:
                    for tok_idx, (tok_start, tok_end) in enumerate(offsets):
                        if tok_start == tok_end:
                            continue
                        if tok_end <= start_char or tok_start >= end_char:
                            continue

                        if labels[tok_idx] == "O":
                            labels[tok_idx] = f"B-{ent_type}"
                        else:
                            labels[tok_idx] = f"I-{ent_type}"

                label_ids = []
                for (tok_start, tok_end), lab in zip(offsets, labels):
                    if tok_start == tok_end:
                        label_ids.append(-100)
                    else:
                        label_ids.append(label_to_id.get(lab, label_to_id["O"]))

                all_labels.append(label_ids)

            encodings.pop("offset_mapping")
            encodings["labels"] = all_labels
            return encodings

        tokenized_ner = raw_datasets.map(
            preprocess_ner,
            batched=True,
            remove_columns=raw_datasets["train"].column_names,
        )

        data_collator = DataCollatorForTokenClassification(tokenizer_ner)

        def compute_metrics(eval_pred):
            predictions, labels = eval_pred
            preds = np.argmax(predictions, axis=-1)

            true_labels = []
            true_preds = []

            for pred_seq, label_seq in zip(preds, labels):
                seq_labels = []
                seq_preds = []
                for p, l in zip(pred_seq, label_seq):
                    if l == -100:
                        continue
                    seq_labels.append(id_to_label[l])
                    seq_preds.append(id_to_label[p])
                true_labels.append(seq_labels)
                true_preds.append(seq_preds)

            return {
                "precision": precision_score(true_labels, true_preds),
                "recall": recall_score(true_labels, true_preds),
                "f1": f1_score(true_labels, true_preds),
                "accuracy": precision_score(true_labels, true_preds),  # Fixed: seqeval doesn't have accuracy_score
            }

        model_safe_name = model_name.replace("/", "--")
        output_dir = f"{base_output_dir}/ner_{model_safe_name}"
        
        ner_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            learning_rate=4e-5,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=epochs,
            weight_decay=0.01,
            logging_steps=1,
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            report_to="none",
            dataloader_pin_memory=False,
            
        )

        ner_trainer = Trainer(
            model=model_ner,
            args=ner_args,
            train_dataset=tokenized_ner["train"],
            eval_dataset=tokenized_ner["validation"],
            data_collator=data_collator,
            tokenizer=tokenizer_ner,
            compute_metrics=compute_metrics,
        )

        print(f"Starting training for {model_name}...")
        train_result = ner_trainer.train()
        
        final_model_dir = f"{base_output_dir}/final_ner_{model_safe_name}"
        ner_trainer.save_model(final_model_dir)
        tokenizer_ner.save_pretrained(final_model_dir)
        
        eval_results = ner_trainer.evaluate()
        
        print(f"Training completed for {model_name}")
        print(f"F1: {eval_results['eval_f1']:.4f}, Precision: {eval_results['eval_precision']:.4f}, Recall: {eval_results['eval_recall']:.4f}")
        
        del model_ner, ner_trainer, tokenized_ner
        cleanup_memory()
        
        return {
            'model_name': model_name,
            'final_model_dir': final_model_dir,
            'metrics': eval_results,
            'label_list': label_list
        }
        
    except Exception as e:
        print(f"Error training {model_name}: {str(e)}")
        return {
            'model_name': model_name,
            'error': str(e),
            'final_model_dir': None,
            'metrics': None
        }

def compare_ner_models(model_candidates, raw_datasets, base_output_dir, max_models=5):
    print("COMPARING NER MODELS")
    print("=" * 50)
    
    results = []
    models_to_try = model_candidates[:max_models]
    
    for model_name in models_to_try:
        result = train_ner_model(
            model_name=model_name,
            raw_datasets=raw_datasets,
            base_output_dir=base_output_dir,
            max_length=512,
            batch_size=16,
            epochs=7
        )
        results.append(result)
    
    successful_models = [r for r in results if r.get('metrics')]
    
    if successful_models:
        successful_models.sort(key=lambda x: x['metrics']['eval_f1'], reverse=True)
        
        print("\nMODEL COMPARISON RESULTS")
        print("=" * 50)
        for i, model_result in enumerate(successful_models):
            metrics = model_result['metrics']
            print(f"{i+1}. {model_result['model_name']}")
            print(f"   F1: {metrics['eval_f1']:.4f} | Precision: {metrics['eval_precision']:.4f} | Recall: {metrics['eval_recall']:.4f}")
        
        best_model = successful_models[0]
        print(f"\nBEST MODEL: {best_model['model_name']}")
        print(f"F1 Score: {best_model['metrics']['eval_f1']:.4f}")
        
        return best_model
    else:
        print("No models trained successfully")
        return None

# Import yang diperlukan untuk metrics
from seqeval.metrics import precision_score, recall_score, f1_score

best_ner_model = compare_ner_models(
    model_candidates=NER_MODEL_CANDIDATES,
    raw_datasets=raw_datasets,
    base_output_dir=BASE_OUTPUT_DIR,
    max_models=4 
)

if best_ner_model:
    print(f"Best NER model saved at: {best_ner_model['final_model_dir']}")
    FINAL_NER_MODEL_PATH = best_ner_model['final_model_dir']
else:
    print("Falling back to default model")
    # Train default model sebagai fallback
    default_result = train_ner_model(
        model_name=NER_MODEL_CANDIDATES[0],
        raw_datasets=raw_datasets,
        base_output_dir=BASE_OUTPUT_DIR
    )
    if default_result.get('final_model_dir'):
        FINAL_NER_MODEL_PATH = default_result['final_model_dir']
    else:
        FINAL_NER_MODEL_PATH = f"{BASE_OUTPUT_DIR}/final_ner_default"
        print(f"Using default path: {FINAL_NER_MODEL_PATH}")

COMPARING NER MODELS
Training NER model: cahya/NusaBert-ner-v1.3


Map:   0%|          | 0/2523 [00:00<?, ? examples/s]

Map:   0%|          | 0/281 [00:00<?, ? examples/s]

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

Original entity types: 56
After normalization: 18
After filtering to target types: 18
Final entity types: 18
Total labels: 37
Labels: ['CRD', 'DAT', 'EVT', 'FAC', 'GPE', 'LAN', 'LAW', 'LOC', 'MON', 'ORD', 'ORG', 'PER', 'PRC', 'PRD', 'QTY', 'REG', 'TIM', 'WOA']


...

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

/tmp/ipykernel_1122/2307255879.py:273: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  ner_trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Starting training for cahya/NusaBert-ner-v1.3...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.352600,0.355265,0.679099,0.661896,0.670387,0.679099
2,0.262500,0.332530,0.693164,0.652636,0.672290,0.693164
3,0.155200,0.350748,0.665308,0.693388,0.679058,0.665308
4,0.118100,0.389361,0.693823,0.661858,0.677464,0.693823
5,0.076900,0.468484,0.695938,0.661399,0.678229,0.695938
6,0.054500,0.560368,0.680716,0.670506,0.675573,0.680716
7,0.039600,0.636550,0.677367,0.665340,0.671300,0.677367


Training completed for cahya/NusaBert-ner-v1.3
F1: 0.6791, Precision: 0.6653, Recall: 0.6934

MODEL COMPARISON RESULTS
1. cahya/NusaBert-ner-v1.3
   F1: 0.6791 | Precision: 0.6653 | Recall: 0.6934

BEST MODEL: cahya/NusaBert-ner-v1.3
F1 Score: 0.6791
Best NER model saved at: ./separated_models/final_ner_cahya--NusaBert-ner-v1.3


In [9]:
# TRAIN CLASSIFICATION MODEL

# Model candidates for classification
CLASS_MODEL_CANDIDATES = [
    # "w11wo/indonesian-roberta-base-sentiment-classifier",
    "indobenchmark/indobert-base-p1",
    # "indolem/indobert-base-uncased",
]

# Category mapping configuration
MAIN_CATS = {
    "finance", "teknologi", "news", "otomotif", "tren", 
    "bola", "lifestyle", "properti", "health", "edukasi", 
    "travel", "lainnya"
}

CATEGORY_MAPPING = {
    "berita-detikhealth": "health",
    "fotohealth": "health",
    "diet": "health",
    "info-sehat": "health",
    "travel-news": "travel",
    "fototravel": "travel",
    "cerita-perjalanan": "travel",
    "domestic-destination": "travel",
    "mie-dan-pasta": "lifestyle",
    "resep-praktis": "lifestyle",
    "sayur": "lifestyle",
    "pengalaman-bersantap": "lifestyle",
    "daging": "lifestyle",
    "tempat-makan": "lifestyle",
    "foto-kuliner": "lifestyle",
    "resto-dan-kafe": "lifestyle",
    "warung-makan": "lifestyle",
    "info-kuliner": "lifestyle",
    "berita-boga": "lifestyle",
    "fotoinet": "teknologi",
    "telecommunication": "teknologi",
    "science": "teknologi",
    "cyberlife": "teknologi",
    "laptop-dan-pc": "teknologi",
    "smartphone": "teknologi",
    "lab-gadget": "teknologi",
    "games-news": "teknologi",
    "law-and-policy": "teknologi",
    "mobile-apps": "teknologi",
    "security": "teknologi",
    "business": "finance",
    "consumer": "finance",
    "sport-lain": "bola",
    "sportstyle": "bola",
    "fotosport": "bola",
    "moto-gp": "otomotif",
    "foto-news": "news",
    "berita": "news",
    "internasional": "news",
    "detiktv": "news",
    "true-story": "news",
    "melindungi-tuah-marwah": "news",
}

def map_category(cat: str) -> str:
    cat = str(cat)
    if cat in CATEGORY_MAPPING:
        return CATEGORY_MAPPING[cat]
    if cat in MAIN_CATS:
        return cat
    return "lainnya"

def calculate_class_weights(raw_train, label2id):
    label_counts = defaultdict(int)

    print("Calculating class weights for classifier...")

    for ex in raw_train:
        lab_id = label2id[map_category(ex["category"])]
        label_counts[lab_id] += 1

    total_samples = sum(label_counts.values())
    num_classes = len(label2id)

    weights = torch.ones(num_classes, dtype=torch.float32)

    for label_id, count in label_counts.items():
        if count > 0:
            weights[label_id] = total_samples / (count * num_classes)

    weights = weights / weights.sum() * num_classes

    print("\nClass distribution:")
    for label_id in sorted(label_counts.keys()):
        label_name = list(label2id.keys())[list(label2id.values()).index(label_id)]
        count = label_counts[label_id]
        freq = count / total_samples
        w = weights[label_id].item()
        print(f"  {label_name:15}: count={count:6d} ({freq:.3%}) | weight={w:.4f}")

    print(f"Total samples: {total_samples}")
    return weights

class WeightedClassificationTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
        if self.class_weights is not None:
            self.class_weights = self.class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None and labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits, labels)
        else:
            loss = outputs.loss

        return (loss, outputs) if return_outputs else loss

def compute_metrics_cls(eval_pred):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

def train_classification_model(model_name, raw_datasets, base_output_dir, batch_size=32, epochs=5):
    print(f"Training classification model: {model_name}")
    print("=" * 50)
    
    try:
        # Apply category mapping
        def map_category_example(example):
            return {"category_mapped": map_category(example["category"])}
        
        mapped_datasets = raw_datasets.map(map_category_example)
        
        # Prepare labels
        label_list = sorted(mapped_datasets["train"].unique("category_mapped"))
        label2id = {l: i for i, l in enumerate(label_list)}
        id2label = {i: l for i, l in enumerate(label_list)}
        
        # Calculate class weights
        class_weights = calculate_class_weights(mapped_datasets["train"], label2id)

        # Load model and tokenizer
        tokenizer_cls = AutoTokenizer.from_pretrained(model_name)
        model_cls = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=len(label_list),
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )

        def preprocess_cls(examples):
            inputs = [f"Entities: {ner_str}" for ner_str in examples["ner"]]
            enc = tokenizer_cls(
                inputs,
                truncation=True,
                max_length=512,
                padding="max_length",
            )
            enc["labels"] = [label2id[cat] for cat in examples["category_mapped"]]
            return enc

        tokenized_cls = mapped_datasets.map(preprocess_cls, batched=True)
        data_collator_cls = DataCollatorWithPadding(tokenizer_cls)

        # Training arguments
        model_safe_name = model_name.split("/")[-1].replace(" ", "_")
        output_dir = f"{base_output_dir}/cls_{model_safe_name}"
        
        cls_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            learning_rate=4e-5,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=epochs,
            logging_steps=1,
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,
            report_to="none",
            warmup_steps=100,
            weight_decay=0.01,
            metric_for_best_model="f1",
            greater_is_better=True,
            dataloader_pin_memory=False,
        )

        # Trainer
        cls_trainer = WeightedClassificationTrainer(
            model=model_cls,
            args=cls_args,
            train_dataset=tokenized_cls["train"],
            eval_dataset=tokenized_cls["validation"],
            data_collator=data_collator_cls,
            processing_class=tokenizer_cls,
            compute_metrics=compute_metrics_cls,
            class_weights=class_weights,
        )

        print(f"Starting training for {model_name}...")
        cls_trainer.train()
        
        # Save model
        final_model_dir = f"{base_output_dir}/final_cls_{model_safe_name}"
        cls_trainer.save_model(final_model_dir)
        tokenizer_cls.save_pretrained(final_model_dir)
        
        # Evaluate
        eval_results = cls_trainer.evaluate()
        
        print(f"Training completed for {model_name}")
        print(f"F1: {eval_results['eval_f1']:.4f}, Accuracy: {eval_results['eval_accuracy']:.4f}, Precision: {eval_results['eval_precision']:.4f}, Recall: {eval_results['eval_recall']:.4f}")
        
        # Cleanup
        del model_cls, cls_trainer, tokenized_cls, tokenizer_cls
        cleanup_memory()
        
        return {
            'model_name': model_name,
            'final_model_dir': final_model_dir,
            'metrics': eval_results,
            'label_list': label_list
        }
        
    except Exception as e:
        print(f"Error training {model_name}: {str(e)}")
        return {
            'model_name': model_name,
            'error': str(e),
            'final_model_dir': None,
            'metrics': None
        }

def compare_classification_models(model_candidates, raw_datasets, base_output_dir, max_models=5):
    print("COMPARING CLASSIFICATION MODELS")
    print("=" * 50)
    
    results = []
    models_to_try = model_candidates[:max_models]
    
    for model_name in models_to_try:
        result = train_classification_model(
            model_name=model_name,
            raw_datasets=raw_datasets,
            base_output_dir=base_output_dir,
            batch_size=16,
            epochs=7
        )
        results.append(result)
    
    successful_models = [r for r in results if r.get('metrics')]
    
    if successful_models:
        successful_models.sort(key=lambda x: x['metrics']['eval_f1'], reverse=True)
        
        print("\nMODEL COMPARISON RESULTS")
        print("=" * 50)
        for i, model_result in enumerate(successful_models):
            metrics = model_result['metrics']
            print(f"{i+1}. {model_result['model_name']}")
            print(f"   F1: {metrics['eval_f1']:.4f} | Accuracy: {metrics['eval_accuracy']:.4f} | Precision: {metrics['eval_precision']:.4f} | Recall: {metrics['eval_recall']:.4f}")
        
        best_model = successful_models[0]
        print(f"\nBEST MODEL: {best_model['model_name']}")
        print(f"F1 Score: {best_model['metrics']['eval_f1']:.4f}")
        
        return best_model
    else:
        print("No models trained successfully")
        return None

# Apply initial category mapping to dataframe
df_processed["category_mapped"] = df_processed["category"].apply(map_category)

# Run model comparison
best_cls_model = compare_classification_models(
    model_candidates=CLASS_MODEL_CANDIDATES,
    raw_datasets=raw_datasets,
    base_output_dir=BASE_OUTPUT_DIR,
    max_models=3
)

if best_cls_model:
    print(f"Best classification model saved at: {best_cls_model['final_model_dir']}")
    FINAL_CLS_MODEL_PATH = best_cls_model['final_model_dir']
else:
    print("Falling back to default model")
    default_result = train_classification_model(
        model_name=CLASS_MODEL_CANDIDATES[0],
        raw_datasets=raw_datasets,
        base_output_dir=BASE_OUTPUT_DIR
    )
    if default_result.get('final_model_dir'):
        FINAL_CLS_MODEL_PATH = default_result['final_model_dir']
    else:
        FINAL_CLS_MODEL_PATH = f"{BASE_OUTPUT_DIR}/final_cls_default"
        print(f"Using default path: {FINAL_CLS_MODEL_PATH}")

COMPARING CLASSIFICATION MODELS
Training classification model: indobenchmark/indobert-base-p1


Map:   0%|          | 0/2523 [00:00<?, ? examples/s]

Map:   0%|          | 0/281 [00:00<?, ? examples/s]

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

Calculating class weights for classifier...

Class distribution:
  bola           : count=   184 (7.293%) | weight=0.7379
  edukasi        : count=   152 (6.025%) | weight=0.8933
  finance        : count=   599 (23.742%) | weight=0.2267
  health         : count=   174 (6.897%) | weight=0.7803
  lainnya        : count=   160 (6.342%) | weight=0.8486
  lifestyle      : count=   137 (5.430%) | weight=0.9911
  news           : count=   424 (16.805%) | weight=0.3202
  otomotif       : count=   176 (6.976%) | weight=0.7715
  properti       : count=   144 (5.707%) | weight=0.9429
  teknologi      : count=   164 (6.500%) | weight=0.8279
  travel         : count=   174 (6.897%) | weight=0.7803
  tren           : count=    35 (1.387%) | weight=3.8793
Total samples: 2523


...

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

Starting training for indobenchmark/indobert-base-p1...


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.182400,1.488334,0.612100,0.524155,0.539099,0.484230
2,1.021900,1.178650,0.661922,0.593436,0.607100,0.580735
3,0.342200,1.059909,0.661922,0.591599,0.622949,0.595217
4,1.246200,1.036173,0.750890,0.659193,0.675562,0.662804
5,0.036400,1.126314,0.775801,0.661824,0.675868,0.665970
6,0.072500,1.200940,0.761566,0.669247,0.671908,0.666234
7,0.017900,1.291043,0.750890,0.649705,0.662675,0.652452


Training completed for indobenchmark/indobert-base-p1
F1: 0.6662, Accuracy: 0.7616, Precision: 0.6692, Recall: 0.6719

MODEL COMPARISON RESULTS
1. indobenchmark/indobert-base-p1
   F1: 0.6662 | Accuracy: 0.7616 | Precision: 0.6692 | Recall: 0.6719

BEST MODEL: indobenchmark/indobert-base-p1
F1 Score: 0.6662
Best classification model saved at: ./separated_models/final_cls_indobert-base-p1


In [10]:
# TRAIN SUMMARIZATION MODEL

# Model candidates for summarization
SUMM_MODEL_CANDIDATES = [
    # "google-t5/t5-base", 
    # "cahya/t5-base-indonesian-summarization-cased",
    "LazarusNLP/IndoNanoT5-base"
]

rouge_metric = evaluate.load("rouge")

def train_summarization_model(model_name, raw_datasets, base_output_dir, batch_size=8, epochs=5):
    print(f"Training summarization model: {model_name}")
    print("=" * 50)
    
    try:
        # Filter dataset for market categories only
        market_train = raw_datasets["train"].filter(
            lambda x: x["category"] in MARKET_CATEGORIES
        )
        market_valid = raw_datasets["validation"].filter(
            lambda x: x["category"] in MARKET_CATEGORIES
        )

        # Load model and tokenizer
        tokenizer_sum = AutoTokenizer.from_pretrained(model_name)
        model_sum = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        def preprocess_sum(examples):
            inputs = [f"summarize for investor: {doc}" for doc in examples["body_text"]]
            model_inputs = tokenizer_sum(
                inputs,
                max_length=512,
                truncation=True,
                padding="max_length",
            )

            with tokenizer_sum.as_target_tokenizer():
                labels = tokenizer_sum(
                    examples["summary"],
                    max_length=80,
                    truncation=True,
                    padding="max_length",
                )

            labels_ids = []
            for seq in labels["input_ids"]:
                labels_ids.append(
                    [(t if t != tokenizer_sum.pad_token_id else -100) for t in seq]
                )

            model_inputs["labels"] = labels_ids
            return model_inputs

        sum_datasets = DatasetDict({
            "train": market_train,
            "validation": market_valid,
        })

        tokenized_sum = sum_datasets.map(
            preprocess_sum,
            batched=True,
            remove_columns=sum_datasets["train"].column_names,
        )

        data_collator_sum = DataCollatorForSeq2Seq(
            tokenizer_sum,
            model=model_sum,
        )

        def compute_metrics_sum(eval_pred):
            preds, labels = eval_pred

            if isinstance(preds, tuple):
                preds = preds[0]

            decoded_preds = tokenizer_sum.batch_decode(preds, skip_special_tokens=True)
            
            labels = np.where(labels != -100, labels, tokenizer_sum.pad_token_id)
            decoded_labels = tokenizer_sum.batch_decode(labels, skip_special_tokens=True)

            # Postprocess text
            decoded_preds = [p.strip() for p in decoded_preds]
            decoded_labels = [l.strip() for l in decoded_labels]

            result_raw = rouge_metric.compute(
                predictions=decoded_preds,
                references=decoded_labels,
                use_stemmer=True,
            )

            result = {k: float(v) for k, v in result_raw.items()}

            prediction_lens = [
                np.count_nonzero(pred != tokenizer_sum.pad_token_id) for pred in preds
            ]
            result["gen_len"] = float(np.mean(prediction_lens))

            return result

        # Training arguments
        model_safe_name = model_name.split("/")[-1].replace(" ", "_")
        output_dir = f"{base_output_dir}/sum_{model_safe_name}"
        
        sum_args = Seq2SeqTrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            learning_rate=4e-5,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=epochs,
            predict_with_generate=True,
            fp16=False,
            logging_steps=1,
            save_strategy="epoch",
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="rouge1",
            greater_is_better=True,
            report_to="none",
        )

        # Trainer
        sum_trainer = Seq2SeqTrainer(
            model=model_sum,
            args=sum_args,
            train_dataset=tokenized_sum["train"],
            eval_dataset=tokenized_sum["validation"],
            data_collator=data_collator_sum,
            processing_class=tokenizer_sum,
            compute_metrics=compute_metrics_sum,
        )

        print(f"Starting training for {model_name}...")
        sum_trainer.train()
        
        # Save model
        final_model_dir = f"{base_output_dir}/final_sum_{model_safe_name}"
        sum_trainer.save_model(final_model_dir)
        tokenizer_sum.save_pretrained(final_model_dir)
        
        # Evaluate
        eval_results = sum_trainer.evaluate()
        
        print(f"Training completed for {model_name}")
        print(f"ROUGE-1: {eval_results['eval_rouge1']:.4f}, ROUGE-2: {eval_results['eval_rouge2']:.4f}, ROUGE-L: {eval_results['eval_rougeL']:.4f}")
        
        # Cleanup
        del model_sum, sum_trainer, tokenized_sum, tokenizer_sum
        cleanup_memory()
        
        return {
            'model_name': model_name,
            'final_model_dir': final_model_dir,
            'metrics': eval_results
        }
        
    except Exception as e:
        print(f"Error training {model_name}: {str(e)}")
        return {
            'model_name': model_name,
            'error': str(e),
            'final_model_dir': None,
            'metrics': None
        }

def compare_summarization_models(model_candidates, raw_datasets, base_output_dir, max_models=5):
    print("COMPARING SUMMARIZATION MODELS")
    print("=" * 50)
    
    results = []
    models_to_try = model_candidates[:max_models]
    
    for model_name in models_to_try:
        result = train_summarization_model(
            model_name=model_name,
            raw_datasets=raw_datasets,
            base_output_dir=base_output_dir,
            batch_size=8,
            epochs=10
        )
        results.append(result)
    
    successful_models = [r for r in results if r.get('metrics')]
    
    if successful_models:
        successful_models.sort(key=lambda x: x['metrics']['eval_rouge1'], reverse=True)
        
        print("\nMODEL COMPARISON RESULTS")
        print("=" * 50)
        for i, model_result in enumerate(successful_models):
            metrics = model_result['metrics']
            print(f"{i+1}. {model_result['model_name']}")
            print(f"   ROUGE-1: {metrics['eval_rouge1']:.4f} | ROUGE-2: {metrics['eval_rouge2']:.4f} | ROUGE-L: {metrics['eval_rougeL']:.4f}")
        
        best_model = successful_models[0]
        print(f"\nBEST MODEL: {best_model['model_name']}")
        print(f"ROUGE-1 Score: {best_model['metrics']['eval_rouge1']:.4f}")
        
        return best_model
    else:
        print("No models trained successfully")
        return None

best_sum_model = compare_summarization_models(
    model_candidates=SUMM_MODEL_CANDIDATES,
    raw_datasets=raw_datasets,
    base_output_dir=BASE_OUTPUT_DIR,
    max_models=4
)

if best_sum_model:
    print(f"Best summarization model saved at: {best_sum_model['final_model_dir']}")
    FINAL_SUM_MODEL_PATH = best_sum_model['final_model_dir']
else:
    print("Falling back to default model")
    FINAL_SUM_MODEL_PATH = f"{BASE_OUTPUT_DIR}/final_sum_default"

COMPARING SUMMARIZATION MODELS
Training summarization model: LazarusNLP/IndoNanoT5-base


Filter:   0%|          | 0/2523 [00:00<?, ? examples/s]

Filter:   0%|          | 0/281 [00:00<?, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Starting training for LazarusNLP/IndoNanoT5-base...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.344200,2.525358,0.253331,0.106152,0.217967,0.217974,20.000000
2,1.925200,2.398138,0.238809,0.091497,0.198517,0.197939,20.000000
3,1.544300,2.323754,0.241291,0.105943,0.206271,0.205554,19.197531
4,1.680300,2.339295,0.266674,0.120729,0.233006,0.233202,19.679012
5,1.646000,2.387329,0.273133,0.128294,0.236414,0.236201,19.839506
6,1.316200,2.465288,0.268157,0.124854,0.234372,0.233800,20.000000
7,1.281000,2.493622,0.276981,0.133973,0.239809,0.239521,19.839506
8,1.028300,2.548248,0.282138,0.135766,0.243426,0.243675,20.000000
9,0.872200,2.573763,0.284613,0.135766,0.247739,0.247594,20.000000
10,0.680200,2.584856,0.295613,0.142929,0.255907,0.255957,20.000000


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Training completed for LazarusNLP/IndoNanoT5-base
ROUGE-1: 0.2956, ROUGE-2: 0.1429, ROUGE-L: 0.2559

MODEL COMPARISON RESULTS
1. LazarusNLP/IndoNanoT5-base
   ROUGE-1: 0.2956 | ROUGE-2: 0.1429 | ROUGE-L: 0.2559

BEST MODEL: LazarusNLP/IndoNanoT5-base
ROUGE-1 Score: 0.2956
Best summarization model saved at: ./separated_models/final_sum_IndoNanoT5-base


In [11]:
from torch.utils.data import Dataset
import json
import re
import ast
from tqdm import tqdm
import evaluate
from sklearn.metrics import classification_report
import torch
import numpy as np

ENTITY_TYPE_MAPPING = {
    "AGE": "QTY", "BEA": "PRD", "COL": "ORG", "CULT": "REG", "CUR": "MON",
    "DAY": "DAT", "HOT": "FAC", "LA": "LAW", "LAN": "LAN", "LAW": "LAW",
    "LEAGUE": "ORG", "NOR": "ORG", "ORD": "ORD", "ORF": "ORG", "ORG": "ORG",
    "PER": "PER", "PLAN": "EVT", "POL": "ORG", "POR": "ORG", "PRC": "PRC",
    "PRD": "PRD", "PRO": "ORG", "PROC": "EVT", "PROD": "PRD", "QTY": "QTY",
    "RAT": "PRC", "REG": "GPE", "REL": "REG", "TECH": "PRD", "TIM": "TIM",
    "URL": "WEB", "WAA": "WOA", "WAO": "WOA", "WEB": "WEB", "WGA": "WOA",
    "WOA": "WOA", "WOK": "WOA", "WOP": "WOA", "WP": "WOA", "WPA": "WOA", "WQA": "WOA",
    "CRD": "CRD", "DAT": "DAT", "EVT": "EVT", "FAC": "FAC", "GPE": "GPE", 
    "LOC": "LOC", "MON": "MON"
}

TARGET_ENTITY_TYPES = {
    "CRD", "DAT", "EVT", "FAC", "GPE", "LAW", "LOC", "MON", 
    "NOR", "ORD", "ORG", "PER", "PRC", "PRD", "QTY", "REG", 
    "TIM", "WOA", "LAN"
}

def map_entity_type(entity_type):
    """Map entity type ke standard type"""
    mapped = ENTITY_TYPE_MAPPING.get(entity_type, entity_type)
    if mapped not in TARGET_ENTITY_TYPES:
        return "ORG"
    return mapped

def parse_ner_field(raw):
    if raw is None:
        return []
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        return [raw]
    if isinstance(raw, str):
        s = raw.strip()
        if not s:
            return []
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list):
                return obj
            if isinstance(obj, dict):
                return [obj]
            return []
        except Exception:
            return []
    return []

def norm_type(t):
    """Normalize type dengan mapping yang konsisten"""
    original_norm = t.upper().replace(" ", "_").replace("/", "_")
    return map_entity_type(original_norm)

def find_entity_spans(text, entities):
    """SAMA PERSIS DENGAN YANG DI TRAINING"""
    spans = []
    for ent in entities:
        ent_text = (ent.get("entity") or "").strip()
        ent_type_raw = ent.get("type")
        if not ent_text or not ent_type_raw:
            continue

        # Gunakan mapped entity type yang SAMA
        ent_type = norm_type(ent_type_raw)
        pattern = re.escape(ent_text)
        for m in re.finditer(pattern, text):
            start, end = m.start(), m.end()
            spans.append((start, end, ent_type))
    return spans

def calculate_ner_metrics_consistent(pipeline, test_df, batch_size=8):
    """Evaluation yang konsisten dengan training approach"""
    
    tokenizer = pipeline.ner_pipe.tokenizer
    all_true_labels = []
    all_pred_labels = []
    
    print("Running consistent NER evaluation...")
    
    for i in tqdm(range(len(test_df)), desc="Evaluating NER"):
        text = str(test_df.iloc[i]["body_text"])
        ner_annotations = test_df.iloc[i]["ner"]
        
        # Preprocess text sama seperti training
        safe_text = truncate_text_by_tokens(tokenizer, text, pipeline.ner_max_len)
        
        # Get gold entities - SAMA dengan training
        gold_ents = parse_ner_field(ner_annotations)
        gold_spans = find_entity_spans(safe_text, gold_ents)
        
        # Convert gold spans to token labels - SAMA dengan training
        encoding = tokenizer(
            safe_text,
            truncation=True,
            padding="max_length", 
            max_length=pipeline.ner_max_len,
            return_offsets_mapping=True,
        )
        offsets = encoding["offset_mapping"]
        
        gold_labels = ["O"] * len(offsets)
        for (start_char, end_char, ent_type) in gold_spans:
            for tok_idx, (tok_start, tok_end) in enumerate(offsets):
                if tok_start == tok_end:
                    continue
                if tok_end <= start_char or tok_start >= end_char:
                    continue
                
                if gold_labels[tok_idx] == "O":
                    gold_labels[tok_idx] = f"B-{ent_type}"
                else:
                    gold_labels[tok_idx] = f"I-{ent_type}"
        
        # Get predictions
        pred_entities = pipeline.ner_pipe(safe_text)
        
        # Convert predictions to token labels
        pred_labels = ["O"] * len(offsets)
        for entity in pred_entities:
            ent_word = entity["word"]
            ent_type = norm_type(entity["entity_group"])
            ent_start = entity["start"]
            ent_end = entity["end"]
            
            for tok_idx, (tok_start, tok_end) in enumerate(offsets):
                if tok_start == tok_end:
                    continue
                if tok_end <= ent_start or tok_start >= ent_end:
                    continue
                
                if pred_labels[tok_idx] == "O":
                    pred_labels[tok_idx] = f"B-{ent_type}"
                else:
                    pred_labels[tok_idx] = f"I-{ent_type}"
        
        # Filter out padding/special tokens (-100 in training)
        filtered_gold = []
        filtered_pred = []
        for (tok_start, tok_end), gold, pred in zip(offsets, gold_labels, pred_labels):
            if tok_start != tok_end:  # Bukan special token
                filtered_gold.append(gold)
                filtered_pred.append(pred)
        
        all_true_labels.append(filtered_gold)
        all_pred_labels.append(filtered_pred)
    
    # Calculate metrics - SAMA dengan training
    flat_true = [label for seq in all_true_labels for label in seq]
    flat_pred = [label for seq in all_pred_labels for label in seq]
    
    precision = precision_score([flat_true], [flat_pred])
    recall = recall_score([flat_true], [flat_pred]) 
    f1 = f1_score([flat_true], [flat_pred])
    
    print(f"\nNER Evaluation Results (Consistent with Training):")
    print(f"   Precision : {precision:.4f}")
    print(f"   Recall    : {recall:.4f}")
    print(f"   F1-Score  : {f1:.4f}")
    
    return {
        "precision": precision,
        "recall": recall, 
        "f1": f1,
        "true_labels": all_true_labels,
        "pred_labels": all_pred_labels
    }

def truncate_text_by_tokens(tokenizer, text, max_len):
    if not isinstance(text, str):
        text = str(text)

    enc = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )
    if len(enc["input_ids"]) <= max_len - 2:
        return text

    low, high = 0, len(text)
    best = 0

    while low < high:
        mid = (low + high) // 2
        sub = text[:mid]
        enc = tokenizer(
            sub,
            add_special_tokens=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )
        if len(enc["input_ids"]) <= max_len - 2:
            best = mid
            low = mid + 1
        else:
            high = mid

    return text[:best] if best > 0 else text[:1]

def gold_entities_to_set(raw_ner):
    ents = parse_ner_field(raw_ner)
    out = set()
    for e in ents:
        name = (e.get("entity") or "").strip().lower()
        t = e.get("type")
        if not name or not t:
            continue
        mapped_type = norm_type(t)
        out.add((name, mapped_type))
    return out

def pred_entities_to_set(pred_list):
    out = set()
    for e in pred_list:
        name = (e.get("word") or "").strip().lower()
        ent_group = e.get("entity_group")
        if not name or not ent_group:
            continue
        mapped_type = norm_type(ent_group)
        out.add((name, mapped_type))
    return out

def calculate_ner_metrics(pred_sets, true_sets):
    tp, fp, fn = 0, 0, 0

    for pred_set, true_set in zip(pred_sets, true_sets):
        this_tp = len(pred_set.intersection(true_set))
        this_fp = len(pred_set) - this_tp
        this_fn = len(true_set) - this_tp

        tp += this_tp
        fp += this_fp
        fn += this_fn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {"Precision": precision, "Recall": recall, "F1-Score": f1}

class NerDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

class SeparatedPipeline:
    def __init__(self, base_dir):
        self.device = 0 if torch.cuda.is_available() else -1

        try:
            self.ner_pipe = pipeline(
                "token-classification",
                model=f"{FINAL_NER_MODEL_PATH}",
                tokenizer=f"{FINAL_NER_MODEL_PATH}",
                aggregation_strategy="simple",
                device=self.device,
            )
            ner_cfg = self.ner_pipe.model.config
            ner_max_pos = getattr(ner_cfg, "max_position_embeddings", None)
            if ner_max_pos is None:
                ner_max_pos = self.ner_pipe.tokenizer.model_max_length
            self.ner_max_len = int(ner_max_pos)

            self.cls_pipe = pipeline(
                "text-classification",
                model=f"{FINAL_CLS_MODEL_PATH}",
                tokenizer=f"{FINAL_CLS_MODEL_PATH}",
                device=self.device,
            )

            self.sum_pipe = pipeline(
                "summarization",
                model=f"{FINAL_SUM_MODEL_PATH}",
                tokenizer=f"{FINAL_SUM_MODEL_PATH}",
                device=self.device,
            )
            sum_cfg = self.sum_pipe.model.config
            sum_max_pos = getattr(sum_cfg, "max_position_embeddings", None)
            if sum_max_pos is None:
                sum_max_pos = self.sum_pipe.tokenizer.model_max_length
            self.sum_max_len = int(sum_max_pos)

        except Exception as e:
            print(f"Gagal memuat model: {e}")
            raise e

        self.target_categories = MARKET_CATEGORIES

    def _ner_output_to_json_string(self, ner_list):
        ents = []
        for e in ner_list:
            name = (e.get("word") or "").strip()
            ent_group = e.get("entity_group")
            if not name or not ent_group:
                continue
            
            mapped_type = norm_type(ent_group)
            
            ents.append({
                "entity": name,
                "type": mapped_type,
            })
        return json.dumps(ents, ensure_ascii=False)

    def run_single(self, text):
        safe_text_for_ner = truncate_text_by_tokens(
            self.ner_pipe.tokenizer,
            text,
            self.ner_max_len,
        )
        ner_raw = self.ner_pipe(safe_text_for_ner)
        ner_str = self._ner_output_to_json_string(ner_raw)

        cls_out = self.cls_pipe(
            f"Entities: {ner_str}",
            truncation=True,
            max_length=64,
        )[0]
        predicted_label = cls_out["label"]

        result = {
            "ner": ner_str,
            "category": predicted_label,
            "summary": "",
            "action": "SKIP",
        }

        if predicted_label in self.target_categories:
            safe_text_for_sum = truncate_text_by_tokens(
                self.sum_pipe.tokenizer,
                text,
                self.sum_max_len,
            )
            result["action"] = "SUMMARIZE"
            sum_out = self.sum_pipe(
                f"summarize for investor: {safe_text_for_sum}",
                max_new_tokens=80,
                min_length=20,
                do_sample=False,
            )[0]["summary_text"]
            result["summary"] = sum_out

        return result

def evaluate_ner_fast(pipeline_instance, test_df, batch_size=8):
    """Gunakan evaluation yang konsisten"""
    
    print("Menggunakan evaluation yang konsisten dengan training...")
    
    results = calculate_ner_metrics_consistent(pipeline_instance, test_df, batch_size)
    
    # Juga tampilkan entity-level comparison untuk debugging
    print("\n" + "="*80)
    print("ENTITY-LEVEL COMPARISON (Debug):")
    print("="*80)
    
    sample_texts = test_df["body_text"].head(3).tolist()
    sample_ner = test_df["ner"].head(3).tolist()
    
    for i, (text, ner_ann) in enumerate(zip(sample_texts, sample_ner)):
        print(f"\nSample {i+1}:")
        print(f"Text: {text[:150]}...")
        
        # Gold entities
        gold_ents = parse_ner_field(ner_ann)
        gold_set = set()
        for ent in gold_ents:
            ent_text = (ent.get("entity") or "").strip()
            ent_type = norm_type(ent.get("type"))
            if ent_text and ent_type:
                gold_set.add((ent_text.lower(), ent_type))
        print(f"Gold entities: {gold_set}")
        
        # Predicted entities
        safe_text = truncate_text_by_tokens(pipeline_instance.ner_pipe.tokenizer, text, pipeline_instance.ner_max_len)
        pred_ents = pipeline_instance.ner_pipe(safe_text)
        pred_set = set()
        for ent in pred_ents:
            ent_text = (ent.get("word") or "").strip()
            ent_type = norm_type(ent.get("entity_group"))
            if ent_text and ent_type:
                pred_set.add((ent_text.lower(), ent_type))
        print(f"Pred entities: {pred_set}")
        
        matches = gold_set.intersection(pred_set)
        print(f"Matches: {len(matches)}/{len(gold_set)}")
        print("-" * 80)

def evaluate_full_pipeline(pipeline_instance, test_df):
    """Evaluation pipeline yang sudah diperbaiki"""
    
    # Pastikan mapping konsisten
    if "category_mapped" not in test_df.columns:
        print("Applying category mapping...")
        test_df = test_df.copy()
        test_df["category_mapped"] = test_df["category"].apply(map_category)
    
    # Evaluasi NER dulu dengan approach konsisten
    ner_results = calculate_ner_metrics_consistent(pipeline_instance, test_df)
    
    # Lanjutkan dengan evaluasi lainnya...
    y_true_cat = []
    y_pred_cat = []
    ref_summaries = []
    pred_summaries = []
    sample_outputs = []

    rouge = evaluate.load("rouge")

    print("Running full pipeline evaluation...")
    for idx, row in tqdm(
        test_df.iterrows(),
        total=len(test_df),
        desc="Processing",
    ):
        text = str(row["body_text"])
        true_cat = str(row["category_mapped"]).strip()
        true_sum = str(row["summary"])

        try:
            output = pipeline_instance.run_single(text)

            y_true_cat.append(true_cat)
            y_pred_cat.append(output["category"].strip())

            if output["category"].strip() in pipeline_instance.target_categories:
                ref_summaries.append(true_sum)
                pred_summaries.append(output["summary"])

            if len(sample_outputs) < 3:
                sample_outputs.append({
                    'text_preview': text[:150] + '...' if len(text) > 150 else text,
                    'true_category': true_cat,
                    'predicted_category': output["category"],
                    'action': output["action"],
                    'summary_preview': output["summary"][:100] + '...' if output["summary"] else 'N/A',
                    'match': true_cat == output["category"].strip()
                })

        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            continue

    # Tampilkan results
    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    
    print(f"\nNER Performance:")
    print(f"   F1-Score  : {ner_results['f1']:.4f}")
    print(f"   Precision : {ner_results['precision']:.4f}")
    print(f"   Recall    : {ner_results['recall']:.4f}")
    
    print("\nClassification Report:")
    if len(y_true_cat) > 0:
        print(classification_report(y_true_cat, y_pred_cat, zero_division=0))
        
        accuracy = sum(1 for true, pred in zip(y_true_cat, y_pred_cat) if true == pred) / len(y_true_cat)
        print(f"Overall Accuracy: {accuracy:.2%}")
    
    print("\nSummarization (ROUGE):")
    if len(ref_summaries) > 0:
        scores = rouge.compute(
            predictions=pred_summaries,
            references=ref_summaries,
        )
        print(f"   ROUGE-1 : {scores['rouge1']*100:.2f}%")
        print(f"   ROUGE-2 : {scores['rouge2']*100:.2f}%")
        print(f"   ROUGE-L : {scores['rougeL']*100:.2f}%")
    else:
        print("   No market categories to summarize")

    cleanup_memory()
    
print("Preparing test data...")
if "category_mapped" not in test_df.columns:
    test_df = test_df.copy()
    test_df["category_mapped"] = test_df["category"].apply(map_category)

pipeline_obj = SeparatedPipeline(BASE_OUTPUT_DIR)

Preparing test data...


Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


In [12]:
# Train
train_df = train_df.copy()
train_df["category_mapped"] = train_df["category"].apply(map_category)
train_df["category"] = train_df["category_mapped"]

evaluate_ner_fast(pipeline_obj, train_df, batch_size=8)
evaluate_full_pipeline(pipeline_obj, train_df)

Menggunakan evaluation yang konsisten dengan training...
Running consistent NER evaluation...


Evaluating NER:   0%|          | 0/2523 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:312: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
Evaluating NER: 100%|██████████| 2523/2523 [03:41<00:00, 11.39it/s]



NER Evaluation Results (Consistent with Training):
   Precision : 0.7153
   Recall    : 0.8291
   F1-Score  : 0.7680

ENTITY-LEVEL COMPARISON (Debug):

Sample 1:
Text: Pelatih tim nasional Italia, Gennaro Gattuso, angkat bicara usai hasil drawing playoff Piala Dunia 2026 Zona Eropa mempertemukan timnya dengan Irlandi...
Gold entities: {('martin dahlin', 'PER'), ('swiss', 'GPE'), ('uefa nations league', 'EVT'), ('20 november 2025', 'DAT'), ('fifa', 'ORG'), ('zurich', 'GPE'), ('italia', 'GPE'), ('kamis', 'DAT'), ('zona eropa', 'GPE'), ('gennaro gattuso', 'PER'), ('makedonia utara', 'GPE'), ('marco materazzi', 'PER'), ('bosnia-herzegovina', 'GPE'), ('football italia', 'ORG'), ('swedia', 'GPE'), ('irlandia utara', 'GPE'), ('path a', 'FAC'), ('wib', 'TIM'), ('wales', 'GPE'), ('piala dunia 2026', 'EVT'), ('20/11/2025', 'DAT'), ('afp', 'ORG')}
Pred entities: {('mat', 'PER'), ('landia', 'GPE'), ('swiss', 'GPE'), ('202', 'EVT'), ('gattus', 'PER'), ('genn', 'PER'), ('league', 'EVT'), ('tiga', '

Evaluating NER: 100%|██████████| 2523/2523 [03:39<00:00, 11.52it/s]



NER Evaluation Results (Consistent with Training):
   Precision : 0.7153
   Recall    : 0.8291
   F1-Score  : 0.7680
Running full pipeline evaluation...


Processing: 100%|██████████| 2523/2523 [05:43<00:00,  7.35it/s]



FINAL RESULTS

NER Performance:
   F1-Score  : 0.7680
   Precision : 0.7153
   Recall    : 0.8291

Classification Report:
              precision    recall  f1-score   support

        bola       0.83      0.03      0.05       184
     edukasi       0.56      0.39      0.46       152
     finance       0.82      0.38      0.52       599
      health       0.32      0.55      0.40       174
     lainnya       0.12      0.55      0.19       160
   lifestyle       0.16      0.47      0.23       137
        news       0.71      0.17      0.28       424
    otomotif       0.82      0.29      0.43       176
    properti       0.59      0.14      0.22       144
   teknologi       0.56      0.37      0.45       164
      travel       0.43      0.55      0.48       174
        tren       0.05      0.17      0.07        35

    accuracy                           0.34      2523
   macro avg       0.50      0.34      0.32      2523
weighted avg       0.60      0.34      0.36      2523

Overall Ac

In [13]:
# Val
val_df = val_df.copy()
val_df["category_mapped"] = val_df["category"].apply(map_category)
val_df["category"] = val_df["category_mapped"]

evaluate_ner_fast(pipeline_obj, val_df, batch_size=8)
evaluate_full_pipeline(pipeline_obj, val_df)

Menggunakan evaluation yang konsisten dengan training...
Running consistent NER evaluation...


Evaluating NER: 100%|██████████| 281/281 [00:24<00:00, 11.33it/s]



NER Evaluation Results (Consistent with Training):
   Precision : 0.6526
   Recall    : 0.7367
   F1-Score  : 0.6921

ENTITY-LEVEL COMPARISON (Debug):

Sample 1:
Text: Apa yang membuat sebuah destinasiwisatasederhana tiba-tiba menjadi magnet baru bagi banyak orang? Apakah panorama alam yang menenangkan, geliat ekonom...
Gold entities: {('bukit idaman', 'FAC'), ('pekon gisting atas', 'GPE'), ('gunung tanggamus', 'LOC'), ('kecamatan gisting', 'GPE'), ('gisting', 'GPE'), ('lampung', 'GPE'), ('tanggamus', 'GPE'), ('700', 'QTY')}
Pred entities: {('isting', 'GPE'), ('pek', 'GPE'), ('on gisting', 'GPE'), ('aman', 'FAC'), ('tanggamus', 'LOC'), ('kecamatan gisting', 'GPE'), ('bukit', 'FAC'), ('atas', 'GPE'), ('meter', 'QTY'), ('lampung', 'GPE'), ('amus', 'GPE'), ('tangg', 'GPE'), ('gunung', 'LOC'), ('id', 'FAC'), ('g', 'GPE'), ('amus', 'LOC'), ('700', 'QTY'), ('tangg', 'LOC')}
Matches: 3/8
--------------------------------------------------------------------------------

Sample 2:
Text: Cristia

Evaluating NER: 100%|██████████| 281/281 [00:24<00:00, 11.30it/s]



NER Evaluation Results (Consistent with Training):
   Precision : 0.6526
   Recall    : 0.7367
   F1-Score  : 0.6921
Running full pipeline evaluation...


Processing: 100%|██████████| 281/281 [00:42<00:00,  6.65it/s]



FINAL RESULTS

NER Performance:
   F1-Score  : 0.6921
   Precision : 0.6526
   Recall    : 0.7367

Classification Report:
              precision    recall  f1-score   support

        bola       0.00      0.00      0.00        29
     edukasi       0.46      0.55      0.50        11
     finance       0.88      0.37      0.52        81
      health       0.36      0.73      0.48        22
     lainnya       0.07      0.29      0.11        17
   lifestyle       0.11      0.60      0.18        10
        news       0.50      0.08      0.14        38
    otomotif       1.00      0.31      0.48        16
    properti       0.50      0.05      0.10        19
   teknologi       0.38      0.19      0.25        16
      travel       0.32      0.30      0.31        20
        tren       0.00      0.00      0.00         2

    accuracy                           0.29       281
   macro avg       0.38      0.29      0.26       281
weighted avg       0.51      0.29      0.31       281

Overall Ac

In [14]:
# Test
test_df = test_df.copy()
test_df["category_mapped"] = test_df["category"].apply(map_category)
test_df["category"] = test_df["category_mapped"]

evaluate_ner_fast(pipeline_obj, test_df, batch_size=8)
evaluate_full_pipeline(pipeline_obj, test_df)

Menggunakan evaluation yang konsisten dengan training...
Running consistent NER evaluation...


Evaluating NER: 100%|██████████| 312/312 [00:28<00:00, 11.07it/s]



NER Evaluation Results (Consistent with Training):
   Precision : 0.6226
   Recall    : 0.7209
   F1-Score  : 0.6682

ENTITY-LEVEL COMPARISON (Debug):

Sample 1:
Text: Strava merilis fitur baru bernama "For a Cause" atau "Untuk Sebuah Tujuan". Dengan fitur ini, pengguna bisa melakukan aktivitas olahraga seperti biasa...
Gold entities: {('bunuh diri', 'PRD'), ('achilles international', 'ORG'), ('untuk sebuah tujuan', 'EVT'), ('indonesia', 'GPE'), ('17/11/2025', 'DAT'), ('kesehatan mental', 'PRD'), ('kanker prostat', 'PRD'), ('kompastekno', 'ORG'), ('for a cause', 'EVT'), ('kanker testis', 'PRD'), ('ms', 'PRD'), ('stravastrava', 'ORG'), ('movember', 'ORG'), ('multiple sclerosis', 'PRD'), ('strava', 'ORG'), ('senin', 'TIM'), ('the national ms society', 'ORG')}
Pred entities: {('kompas', 'ORG'), ('international', 'ORG'), ('/', 'DAT'), ('a', 'WOA'), ('ember', 'ORG'), ('testis', 'PRD'), ('5', 'DAT'), ('202', 'DAT'), ('untuk', 'WOA'), ('sc', 'PRD'), ('multi', 'PRD'), ('strav', 'ORG'), ('mov'

Evaluating NER: 100%|██████████| 312/312 [00:28<00:00, 11.11it/s]



NER Evaluation Results (Consistent with Training):
   Precision : 0.6226
   Recall    : 0.7209
   F1-Score  : 0.6682
Running full pipeline evaluation...


Processing: 100%|██████████| 312/312 [00:47<00:00,  6.56it/s]



FINAL RESULTS

NER Performance:
   F1-Score  : 0.6682
   Precision : 0.6226
   Recall    : 0.7209

Classification Report:
              precision    recall  f1-score   support

        bola       0.00      0.00      0.00        27
     edukasi       0.67      0.33      0.44        12
     finance       0.74      0.31      0.44        94
      health       0.29      0.26      0.27        23
     lainnya       0.06      0.46      0.10        13
   lifestyle       0.15      0.69      0.25        13
        news       0.80      0.07      0.13        55
    otomotif       0.50      0.17      0.25        18
    properti       0.20      0.06      0.09        17
   teknologi       0.50      0.33      0.40        21
      travel       0.30      0.33      0.32        18
        tren       0.00      0.00      0.00         1

    accuracy                           0.24       312
   macro avg       0.35      0.25      0.22       312
weighted avg       0.51      0.24      0.27       312

Overall Ac

In [15]:
# UJI COBA

# Contoh Teks 1: Berita Umum (Harus diskip summary-nya)
sample_text_general = """
Harga aneka smartphone Xiaomi yang dipasarkan tahun depan akan mengalami kenaikan. Pernyataan ini disampaikan oleh Presiden Xiaomi, Lu Weibing dalam konferensi pers terkait laporan pendapatan perusahaan.
Kenaikan hargaHPXiaomidipicu oleh meningkatnya harga chip memori, seiring dengan lonjakan permintaan komponen ini untuk server kecerdasan buatan (AI) yang diperlukan oleh perusahaan teknologi untuk membangun data center.
Tingginya permintaan chip memori untuk server juga membuat perusahaan seperti Samsung, memangkas produksi chip memori termasuk untuk ponsel, dan mengalihkannya ke memori bandwidth tinggi (high bandwidth memory).
Karena kondisi itu, Weibing lantas memperingatkan pengguna akan kenaikan harga ponsel Xiaomi.
""Saya memperkirakan tekanan akan jauh lebih berat tahun depan dibanding tahun ini,"" kata Weibing, dikutipKompasTeknodariReuters.
""Secara umum, konsumen mungkin akan mendapati kenaikan harga ecer produk yang cukup besar. Sebagian tekanan mungkin harus diatasi melalui kenaikan harga, walaupun cara ini saja tidak akan cukup untuk mengatasinya,"" jelas bos Xiaomi itu.
Bukan kali ini saja, bulan lalu Weibing juga memaparkan bahwa melonjaknya harga chip memori mendesak kenaikan hargasmartphone. Pernyataan ini mencuat usai sejumlah konsumen mengaku kecewa dengan harga Redmi K90.
KOMPAS.com/Caroline Saskia TanotoIlustrasi smartphone yang cukup diminati di ITC Roxy Mas
Ponsel andalan dari sub-merek Xiaomi itu dibanderol 2.599 yuan (sekitar Rp 6,1 juta) untuk versi dasarnya dengan RAM 12/256 GB. Harganya naik dari 2.499 yuan (sekitar Rp 5,8 juta) dibanding pendahulunya, Redmi K80 yang rilis November 2024 lalu.
Walau memberikan peringatan kenaikan harga, Weibing tak merinci persentase lonjakanharga HP Xiaomidi tahun depan. Tak dirinci pula apakah lonjakan harga HP Xiaomi akan berlaku secara global termasuk Indonesia, atau hanya di pasar tertentu saja.
Di tengah tekanan harga komponen, Xiaomi tetap catatkan pertumbuhan pengiriman. Laporan dari firma riset pasar Omdia menyebut Xiaomi berhasil mengirimkan 43,4 juta unit ponsel pada kuartal III-2025.
Angka tersebut mencerminkan kenaikan 1 persen dibandingkan periode yang sama tahun lalu. Dengan total pengiriman tersebut, Xiaomi mampu meraih pangsa pasar smartphone global sebesar 14 persen, dan berada di posisi ketiga.
Posisi pertama adalah Samsung dengan pengiriman 60,6 juta unit ponsel (market share 19 persen), disusul Apple sebesar 56,5 juta unit (pangsa pasar 18 persen).
Posisi keempat ada Transsion dengan pengiriman 28,6 juta unit (pangsa pasar 9 persen), yang disusul Vivo dengan pengiriman 28,5 juta unit (pangsa pasar 9 persen)."""

# Contoh Teks 2: Berita Saham (Harus diringkas)
sample_text_market = """
PT Bank Syariah Indonesia Tbk (BSI) menjalin kerja sama dengan Kementerian Haji dan Umrah Republik Indonesia (Kemenhaj) untuk memperkuat digitalisasi layanan keuangan bagi jamaah haji asal Indonesia. Penandatanganan nota kesepahaman tersebut dilakukan di Jakarta dan menjadi bagian dari persiapan penyelenggaraan haji 2026.
Kesepakatan ini ditandatangani oleh Direktur UtamaBSIAnggoro Eko Cahyo dan Menteri Haji dan Umrah Mochamad Irfan Yusuf. Kedua pihak menyetujui kolaborasi dalam operasional transaksi kelembagaan, penyediaan akses layanan haji bagi calon jamaah, serta pemanfaatan produk keuangan syariah BSI.
Penandatanganan ini sekaligus menandai dimulainya tahap awal layanan haji 2026. Pemerintah menetapkan Biaya Penyelenggaraan Ibadah Haji (Bipih) 2026 sebesar Rp 87.409.365,45 per jamaah.
Rata-rata jamaah membayar Rp 54.193.806,58, sementara sisanya ditutup dari nilai manfaat dana haji. Bipih tahun ini lebih rendah sekitar Rp 2 juta dibandingkan tahun sebelumnya, dan proses pelunasan diperkirakan dimulai pada pekan keempat November 2025.
Direktur Utama BSI Anggoro Eko Cahyo menyatakan apresiasinya terhadap kerja sama tersebut karena dinilai memperkuat layanan bagi jamaah Indonesia yang menjadi salah satu rombongan terbesar setiap tahun.
“Kami berkomitmen menyediakan layanan perbankan syariah yang lebih cepat, aman, dan terkoneksi dengan sistem di Arab Saudi sehingga jamaah dapat beribadah dengan tenang dan juga dapat melakukan transaksi keuangan di Arab Saudi secara aman,” ujarnya, melalui keterangan pers, dikutip Sabtu (22/11/2025).
BSI telah menyiapkan berbagai kanal untuk pelunasan biaya haji, baik secara offline, online, maupun melalui BSI Agen yang tersebar di seluruh Indonesia.
Menteri Haji dan Umrah Mochamad Irfan Yusuf menegaskan komitmen pemerintah untuk terus meningkatkan kualitas layanan bagi jamaah Indonesia.
“Sebagai salah satu negara dengan jumlah penduduk Muslim terbesar di dunia, Indonesia memiliki posisi strategis dalam penyelenggaraan ibadah haji, sehingga peningkatan pelayanan haji dan umrah menjadi prioritas utama kami setiap tahun,” ujarnya.
Ia berharap kolaborasi dengan BSI sebagai bank syariah terbesar dapat memberi kemudahan tambahan bagi jamaah. “Kami berharap BSI juga dapat mempermudah jamaah baik di Tanah Air maupun saat berada di Tanah Suci,” katanya.
BSI saat ini menjadi market leader layanan perbankan jamaah haji. Rata-rata 83 persen calon haji Indonesia menggunakan layanan BSI untuk keberangkatan ke Arab Saudi. Di sisi tabungan haji, BSI mengelola lebih dari 6,7 juta rekening, dengan sekitar 51 persen di antaranya sudah masuk daftar tunggu.
"""

print("\n=== TEST 1: MARKET NEWS ===")
res1 = pipeline_obj.run_single(sample_text_market)
print(json.dumps(res1, indent=2))

print("\n=== TEST 2: GENERAL NEWS ===")
res2 = pipeline_obj.run_single(sample_text_general)
print(json.dumps(res2, indent=2))


=== TEST 1: MARKET NEWS ===
{
  "ner": "[{\"entity\": \"PT\", \"type\": \"ORG\"}, {\"entity\": \"Bank\", \"type\": \"ORG\"}, {\"entity\": \"Syariah\", \"type\": \"ORG\"}, {\"entity\": \"Indonesia\", \"type\": \"GPE\"}, {\"entity\": \"Tbk\", \"type\": \"ORG\"}, {\"entity\": \"B\", \"type\": \"ORG\"}, {\"entity\": \"SI\", \"type\": \"ORG\"}, {\"entity\": \"Kementerian\", \"type\": \"ORG\"}, {\"entity\": \"Haji\", \"type\": \"ORG\"}, {\"entity\": \"dan\", \"type\": \"ORG\"}, {\"entity\": \"Umrah\", \"type\": \"ORG\"}, {\"entity\": \"Republik\", \"type\": \"ORG\"}, {\"entity\": \"Indonesia\", \"type\": \"GPE\"}, {\"entity\": \"Kemen\", \"type\": \"ORG\"}, {\"entity\": \"h\", \"type\": \"ORG\"}, {\"entity\": \"aj\", \"type\": \"ORG\"}, {\"entity\": \"Indonesia\", \"type\": \"GPE\"}, {\"entity\": \"Jakarta\", \"type\": \"GPE\"}, {\"entity\": \"202\", \"type\": \"DAT\"}, {\"entity\": \"6\", \"type\": \"DAT\"}, {\"entity\": \"B\", \"type\": \"ORG\"}, {\"entity\": \"SI\", \"type\": \"ORG\"}, {